In [6]:
# ============================================================
# NEURAL STRESS DETECTION MODEL
# BiGRU Prosody / Stress Prediction
# ============================================================

# =========================
# INSTALLS
# =========================

!pip install pronouncing
!pip install nltk

# =========================
# IMPORTS
# =========================

import pronouncing
import nltk
import re
import random
import torch
import torch.nn as nn

from nltk.corpus import gutenberg
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from nltk.tokenize import PunktSentenceTokenizer

# =========================
# DOWNLOAD DATA
# =========================

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

# =========================
# LOAD TEXT DATA
# =========================

# Using Shakespeare + other literary sources
files = [
    'shakespeare-hamlet.txt',
    'melville-moby_dick.txt',
    'austen-emma.txt'
]

lines = []

for file in files:

    sents = gutenberg.sents(file)

    for sent in sents:

        line = " ".join(sent)

        # Keep medium-length lines
        if 4 <= len(sent) <= 20:
            lines.append(line)

# Shuffle dataset
random.shuffle(lines)

# Limit size for Colab speed
lines = lines[:5000]

print("Total lines:", len(lines))

# =========================
# CLEAN TEXT
# =========================

def clean_text(text):

    text = text.lower()

    text = re.sub(r"[^a-zA-Z\s]", "", text)

    text = re.sub(r"\s+", " ", text).strip()

    return text

# =========================
# STRESS EXTRACTION
# =========================

# 0 = unstressed
# 1 = primary stress
# 2 = secondary stress

def get_word_stress(word):

    phones = pronouncing.phones_for_word(word)

    if len(phones) == 0:

        return [0]

    stresses = pronouncing.stresses(phones[0])

    return [int(s) for s in stresses]

# =========================
# WORD-LEVEL LABELS
# =========================

# We simplify the problem:
# Each word gets ONE label
#
# Rule:
# If a word contains primary stress -> 1
# Else if secondary stress -> 2
# Else -> 0

def compress_stress(stress_list):

    if 1 in stress_list:
        return 1

    elif 2 in stress_list:
        return 2

    else:
        return 0

# =========================
# PREPROCESS DATA
# =========================

processed_data = []

for line in lines:

    clean_line = clean_text(line)

    words = clean_line.split()

    if len(words) < 2:
        continue

    stress_labels = []

    valid = True

    for word in words:

        stress_pattern = get_word_stress(word)

        if len(stress_pattern) == 0:
            valid = False
            break

        label = compress_stress(stress_pattern)

        stress_labels.append(label)

    if valid:

        processed_data.append((words, stress_labels))

print("Processed examples:", len(processed_data))

# =========================
# BUILD VOCAB
# =========================

all_words = []

for words, labels in processed_data:

    all_words.extend(words)

word_counter = Counter(all_words)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in word_counter:

    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

# =========================
# ENCODE WORDS
# =========================

def encode_words(words):

    return [
        vocab.get(word, vocab["<UNK>"])
        for word in words
    ]

# =========================
# TRAIN / TEST SPLIT
# =========================

train_data, test_data = train_test_split(
    processed_data,
    test_size=0.2,
    random_state=42
)

# =========================
# DATASET CLASS
# =========================

class StressDataset(Dataset):

    def __init__(self, data):

        self.data = data

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        words, labels = self.data[idx]

        x = torch.tensor(
            encode_words(words),
            dtype=torch.long
        )

        y = torch.tensor(
            labels,
            dtype=torch.long
        )

        return x, y

# =========================
# COLLATE FUNCTION
# =========================

def collate_fn(batch):

    xs = [item[0] for item in batch]
    ys = [item[1] for item in batch]

    xs_padded = pad_sequence(
        xs,
        batch_first=True,
        padding_value=0
    )

    ys_padded = pad_sequence(
        ys,
        batch_first=True,
        padding_value=-100
    )

    return xs_padded, ys_padded

# =========================
# DATALOADERS
# =========================

train_dataset = StressDataset(train_data)
test_dataset = StressDataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

# =========================
# DEVICE
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

# =========================
# MODEL
# =========================

class StressModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=128,
        hidden_dim=256
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=0
        )

        self.bigru = nn.GRU(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(
            hidden_dim * 2,
            3
        )

    def forward(self, x):

        x = self.embedding(x)

        outputs, _ = self.bigru(x)

        logits = self.fc(outputs)

        return logits

# =========================
# INITIALIZE MODEL
# =========================

model = StressModel(
    vocab_size=len(vocab)
).to(device)

# =========================
# LOSS + OPTIMIZER
# =========================

criterion = nn.CrossEntropyLoss(ignore_index=-100)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# =========================
# TRAINING LOOP
# =========================

epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for x_batch, y_batch in train_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(x_batch)

        loss = criterion(
            logits.view(-1, 3),
            y_batch.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

# =========================
# EVALUATION
# =========================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for x_batch, y_batch in test_loader:

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)

        predictions = torch.argmax(logits, dim=-1)

        mask = y_batch != -100

        correct += (
            (predictions == y_batch) * mask
        ).sum().item()

        total += mask.sum().item()

accuracy = correct / total

print("\nTest Accuracy:", round(accuracy, 4))

# =========================
# LABEL MAPPING
# =========================

label_map = {
    0: "unstressed",
    1: "primary",
    2: "secondary"
}

# =========================
# PREDICTION FUNCTION
# =========================

def predict_stress(line):

    model.eval()

    clean_line = clean_text(line)

    words = clean_line.split()

    encoded = torch.tensor(
        [encode_words(words)],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        logits = model(encoded)

        predictions = torch.argmax(
            logits,
            dim=-1
        )

    predictions = predictions[0].cpu().numpy()

    print("\nINPUT:")
    print(line)

    print("\nPREDICTIONS:")

    for word, pred in zip(words, predictions):

        print(
            f"{word:15s} -> {label_map[int(pred)]}"
        )

# =========================
# TEST EXAMPLES
# =========================

predict_stress(
    "Shall I compare thee to a summer day"
)

predict_stress(
    "Once upon a midnight dreary while I pondered weak and weary"
)

predict_stress(
    "Because I could not stop for Death"
)

[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total lines: 5000
Processed examples: 4991
Vocabulary size: 6204
Using device: cpu
Epoch 1 Loss: 0.2016
Epoch 2 Loss: 0.1066
Epoch 3 Loss: 0.0801
Epoch 4 Loss: 0.0532
Epoch 5 Loss: 0.0299

Test Accuracy: 0.9702

INPUT:
Shall I compare thee to a summer day

PREDICTIONS:
shall           -> primary
i               -> primary
compare         -> primary
thee            -> primary
to              -> primary
a               -> unstressed
summer          -> primary
day             -> primary

INPUT:
Once upon a midnight dreary while I pondered weak and weary

PREDICTIONS:
once            -> primary
upon            -> primary
a               -> unstressed
midnight        -> primary
dreary          -> primary
while           -> primary
i               -> primary
pondered        -> primary
weak            -> primary
and             -> unstressed
weary           -> primary

INPUT:
Because I could not stop for Death

PREDICTIONS:
because         -> unstressed
i               -> primary
could       

In [8]:
predict_stress(
    "The quick brown fox jumps over the lazy dog"
)


INPUT:
The quick brown fox jumps over the lazy dog

PREDICTIONS:
the             -> unstressed
quick           -> primary
brown           -> primary
fox             -> primary
jumps           -> primary
over            -> primary
the             -> unstressed
lazy            -> primary
dog             -> primary
